<!-- CONCLUSION-CELL -->
> ## ✅ 결론 — Feature 선택: **전체 유지 (공격적 제거 안 함)**
>
> | 실험 | feature 수 | Val R² | Test R² |
> |---|---|---|---|
> | **E6-0 전체(기준)** | 59 | **0.0881** | 0.0344 |
> | E6-1 하위20% 제거 | 47 | 0.0876 | 0.0363 |
> | E6-2 하위40% 제거 | 35 | 0.0864 | 0.0288 |
> | E6-3a 그룹별만 | 11 | 0.0606 | 0.0408 |
>
> - 하위 feature 제거해도 Val R² **개선 없음**(오히려 약간 하락).
> - 하위20% 제거(47개)는 거의 동등 → 경량화 옵션으로만 의미.
> - **→ 전체 59 feature 유지를 기본으로 함.**


# 11. Feature 선택 실험 (Phase 8)
- **선행 조건**: 10_tuning_experiment 완료 후 최적 파라미터 아래 셀에 입력
- 튜닝된 모델 기반 SHAP 재계산 → 저중요 feature 제거 실험

| 실험 | 내용 |
|---|---|
| E6-1 | SHAP 하위 20% feature 제거 |
| E6-2 | SHAP 하위 40% feature 제거 |
| E6-3 | 구종별 vs 그룹별 feature 비교 |
| E6-4 | 최적 feature set 확정 |

In [ ]:
import os, sys

IN_COLAB = os.path.exists('/content')

if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/투수 컨디션 예측 ML'
else:
    DRIVE = r'c:\Users\suyou\OneDrive\Desktop\ASAC\PROJECT\투수 컨디션 예측'

FEATURE_PATH = os.path.join(DRIVE, '0_data', '4_features', 'features_pitch15.parquet')
OUTPUT_DIR   = os.path.join(DRIVE, '4_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'환경: {"코랩" if IN_COLAB else "로컬"}')
print(f'데이터: {FEATURE_PATH}')

In [ ]:
try:
    import xgboost, shap
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'xgboost', 'shap', '-q'])

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('모듈 로드 완료')

## ★ Phase 7 완료 후 최적 파라미터 입력 (필수)

In [ ]:
import json

json_path = os.path.join(OUTPUT_DIR, 'best_params.json')

if os.path.exists(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        saved = json.load(f)
    FINAL_MODEL = saved.get('FINAL_MODEL', 'XGB')
    raw_params  = saved.get(FINAL_MODEL, saved.get('XGB', {}))
    BEST_PARAMS = {
        **raw_params,
        'random_state': 42, 'n_jobs': -1, 'verbosity': 0,
        'early_stopping_rounds': 50,
    }
    print(f'best_params.json 로드 완료 (모델: {FINAL_MODEL})')
    print(json.dumps(raw_params, indent=2))
else:
    # fallback: 10번 미실행 시 수동 입력
    print('⚠️  best_params.json 없음 — 10_tuning_experiment 먼저 실행하세요')
    BEST_PARAMS = {
        'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1,
        'reg_alpha': 0.0, 'reg_lambda': 1.0,
        'random_state': 42, 'n_jobs': -1, 'verbosity': 0,
        'early_stopping_rounds': 50,
    }

In [ ]:
df = pd.read_parquet(FEATURE_PATH)

META_COLS    = ['game_pk', 'pitcher', 'season', 'y_whiff']
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]

train = df[df['season'].isin([2021, 2022, 2023])]
val   = df[df['season'] == 2024]
test  = df[df['season'] == 2025]

X_train, y_train = train[FEATURE_COLS], train['y_whiff']
X_val,   y_val   = val[FEATURE_COLS],   val['y_whiff']
X_test,  y_test  = test[FEATURE_COLS],  test['y_whiff']

print(f'Train: {len(train):,}  Val: {len(val):,}  Features: {len(FEATURE_COLS)}')

## 1. 튜닝 모델 SHAP 분석

In [ ]:
model = xgb.XGBRegressor(**BEST_PARAMS)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

val_r2 = r2_score(y_val, model.predict(X_val))
print(f'튜닝 모델 Val R²: {val_r2:.4f}')

explainer   = shap.TreeExplainer(model)
shap_values = explainer(X_val)
shap_mean   = pd.Series(np.abs(shap_values.values).mean(axis=0), index=FEATURE_COLS).sort_values(ascending=False)

print(f'\n상위 20개 feature:')
print(shap_mean.head(20).round(5))

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_val, plot_type='bar', show=False, max_display=30)
plt.title('튜닝 모델 SHAP Feature Importance')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'shap_tuned_bar.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature 제거 실험 (E6-1 ~ E6-3)

In [ ]:
def train_eval(feature_cols, label=''):
    m = xgb.XGBRegressor(**BEST_PARAMS)
    m.fit(X_train[feature_cols], y_train,
          eval_set=[(X_val[feature_cols], y_val)], verbose=False)
    vr2  = r2_score(y_val,  m.predict(X_val[feature_cols]))
    tr2  = r2_score(y_test, m.predict(X_test[feature_cols]))
    vrmse = mean_squared_error(y_val, m.predict(X_val[feature_cols])) ** 0.5
    print(f'[{label}] features={len(feature_cols)}  Val R²={vr2:.4f}  RMSE={vrmse:.4f}  Test R²={tr2:.4f}')
    return vr2, vrmse, tr2

n_total = len(shap_mean)

# E6-1: 하위 20% 제거
keep80 = shap_mean.head(int(n_total * 0.8)).index.tolist()
# E6-2: 하위 40% 제거
keep60 = shap_mean.head(int(n_total * 0.6)).index.tolist()
# E6-3: 구종별(avg_*_Fastball 등) vs 그룹(avg_speed_all 등) feature만
group_cols  = [c for c in FEATURE_COLS if not any(s in c for s in ['Fastball','Breaking','Offspeed'])]
typed_cols  = [c for c in FEATURE_COLS if any(s in c for s in ['Fastball','Breaking','Offspeed'])]

results = []
for name, cols in [
    ('E6-0 전체 (기준)',    FEATURE_COLS),
    ('E6-1 하위20% 제거',   keep80),
    ('E6-2 하위40% 제거',   keep60),
    ('E6-3a 그룹별만',      group_cols),
    ('E6-3b 구종별만',      typed_cols),
]:
    vr2, vrmse, tr2 = train_eval(cols, name)
    results.append({'name': name, 'n_features': len(cols),
                    'val_r2': vr2, 'val_rmse': vrmse, 'test_r2': tr2})

res_df = pd.DataFrame(results)
print('\n=== 결과 요약 ===')
print(res_df.round(4).to_string(index=False))

## 3. E6-4 최적 feature set 확정

In [ ]:
# 위 결과 보고 최적 feature set 선택 (기본: 전체)
FINAL_FEATURE_COLS = FEATURE_COLS   # ← 결과 보고 교체

final_model = xgb.XGBRegressor(**BEST_PARAMS)
final_model.fit(X_train[FINAL_FEATURE_COLS], y_train,
                eval_set=[(X_val[FINAL_FEATURE_COLS], y_val)], verbose=False)

print(f'최종 feature 수: {len(FINAL_FEATURE_COLS)}')
print(f'Val  R²={r2_score(y_val, final_model.predict(X_val[FINAL_FEATURE_COLS])):.4f}')
print(f'Test R²={r2_score(y_test, final_model.predict(X_test[FINAL_FEATURE_COLS])):.4f}')

import joblib
joblib.dump(final_model, os.path.join(OUTPUT_DIR, 'statcast_final_model.pkl'))
pd.Series(FINAL_FEATURE_COLS).to_csv(os.path.join(OUTPUT_DIR, 'final_feature_cols.csv'), index=False)

res_df.to_csv(os.path.join(OUTPUT_DIR, 'feature_selection_results.csv'), index=False)
print('\n저장 완료: statcast_final_model.pkl / final_feature_cols.csv / feature_selection_results.csv')